# Figure 1

In this notebook, we'll generate figure panels for use in Figure 1.

## Load packages

hise: The Human Immune System Explorer R SDK package  
purrr: Functional programming tools  
dplyr: Dataframe handling functions  
ggplot2: plotting functions  
ggrastr: raster plotting for ggplot2  
cowplot: multi-panel plot assembly  
Seurat: single-cell genomics methods  
ArchR: single-cell ATAC-seq methods

In [ ]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(purrr)
quiet_library(dplyr)
quiet_library(ggplot2)
quiet_library(ggrastr)
quiet_library(cowplot)
quiet_library(Seurat)
quiet_library(ArchR)

Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘ggrastr’ was built under R version 4.4.3”
Warning message:
“package ‘cowplot’ was built under R version 4.4.3”
Warning message:
“package ‘Seurat’ was built under R version 4.4.3”
Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Warning message:
“package ‘sp’ was built under R version 4.4.3”


In [ ]:
out_dir <- "output/figure_1"
if(!dir.exists(out_dir)) { dir.create(out_dir, recursive = TRUE) }

## Retrieve files

Now, we'll use the HISE SDK package to retrieve the Seurat objects and cell type labels based on file UUIDs. This will be placed in the `cache/` subdirectory by default.

In [ ]:
so_uuids <- list(
    "7bdac6ef-e5e5-4150-b4f3-9c1a1e250334", # CD4 T cell Seurat object
    "46438bc4-cde6-4ae6-b349-9c513dd9d16f" # CD8 T cell Seurat object
)

In [ ]:
so_files <- cacheFiles(so_uuids)

In [ ]:
label_uuids <- list(
    "ebd4bee7-2f5d-46e1-b2fc-22157f1b8d04", # CD4 type labels
    "4d6aade9-288c-452f-8f0d-ac59e539f4cc"  # CD8 type labels
)

In [ ]:
label_files <- cacheFiles(label_uuids)

In [ ]:
arrow_uuids <- list(
    "052b769d-cbdf-41f6-8fe8-0d34564b442a",
    "16e0c562-5d36-431f-bb27-b443aabc7077",
    "30167a93-70a8-4c38-b615-2252dabe417e",
    "57dde81e-bdaa-4138-add6-2551968672f4",
    "6853fb68-fa85-43d5-9071-c5e42667a75e",
    "6d8185bf-8a35-492d-a6ab-5783006b3b8e",
    "8c2a93be-de53-4d6f-ae2e-ab40a356edc3",
    "9ab975f8-7763-4892-96a7-a7438ecc9470",
    "a56fd2ba-a055-4ff8-9ab1-69df113bc032",
    "ad2f347d-e961-4a70-b294-4df83c12355d",
    "c299a55a-3325-4eb3-ba8e-c8ceccafaa8c",
    "ff8fe67e-cfe0-482a-bad1-aa189390a1c0"
)

In [ ]:
arrow_files <- cacheFiles(arrow_uuids)

## UMAP Projections

To generate UMAPs that provide an overview of the effect of treatment, we'll sample an equal number of cells from each condition.

We'll use this sampling to select cells from our scATAC-seq data, then perform ArchR's IterativeLSI method to obtain reduced dimensions for scATAC.

Then, we'll perform dimensionality reduction using Seurat's methods for scRNA-seq and ADT data.

Once we have all 3 modalities, we can integrate them using Seurat's Weighted Nearest Neighbors methods to generate a joint UMAP projection.

### Select cells

In [ ]:
all_labels <- map_dfr(label_files, read.csv)

For this figure, we'll leave out the untreated cells

In [ ]:
all_labels <- all_labels %>%
  filter(treatment != "untreated")

In [ ]:
sample_counts <- all_labels %>%
    group_by(treatment, timepoint) %>%
    dplyr::summarise(n_cells = n(),
                     .groups = "keep")

In [ ]:
sample_counts

The sample with the fewest cells is DMSO at 72 hr. To balance the samples, let's use 2,500 cells per sample for visualization.

In [ ]:
n_downsample <- 2500

In [ ]:
set.seed(3030)
sampled_labels <- all_labels %>%
  group_by(treatment, timepoint) %>%
  sample_n(n_downsample) %>%
  ungroup()

In [ ]:
nrow(all_labels)
nrow(sampled_labels)

Split cells by timepoint

In [ ]:
split_labels <- split(sampled_labels, sampled_labels$timepoint)

### Run ArchR LSI

In [ ]:
addArchRThreads(12)
addArchRGenome("hg38")
addArchRVerbose(FALSE)

In [ ]:
all_proj <- ArchRProject(
    ArrowFiles = arrow_files,
    copyArrows = TRUE,
    showLogo = FALSE
)

In [ ]:
split_proj <- map2(
    split_labels, names(split_labels),
    function(sampled_labels, tp) {
        archr_meta <- as.data.frame(getCellColData(all_proj))
        sampled_archr_meta <- archr_meta %>%
          mutate(archr_name = rownames(archr_meta)) %>%
          filter(barcodes %in% sampled_labels$barcodes)
        
        sampled_proj <- subsetArchRProject(
            all_proj,
            cells = sampled_archr_meta$archr_name,
            outputDirectory = paste0("SampledArchR_", tp)
        )
    }
)

In [ ]:
split_proj <- map(split_proj, addIterativeLSI)

Extract LSI dimensions and match the order of cells to the sampled metadata

In [ ]:
lsi_mats <- map2(
    split_proj, split_labels,
    function(sampled_proj, sampled_labels) {
        lsi_mat <- getReducedDims(sampled_proj)
        rownames(lsi_mat) <- sub(".+#", "", rownames(lsi_mat))
        lsi_mat <- lsi_mat[sampled_labels$barcodes,]
        lsi_mat
    }
)

Save LSI results

In [ ]:
walk2(
    lsi_mats, names(split_labels),
    function(lsi_mat, tp) {
        saveRDS(
            lsi_mat,
            file.path(out_dir, paste0("atac_lsi_mat_", tp, "_", Sys.Date(), ".rds"))
        )
    }
)

### Run Seurat PCA for scRNA

In [ ]:
cd4_so <- readRDS(so_files[1])
all_cd4_meta <- cd4_so@meta.data
cd4_so <- cd4_so[,cd4_so$barcodes %in% sampled_labels$barcodes]
cd8_so <- readRDS(so_files[2])
all_cd8_meta <- cd4_so@meta.data
cd8_so <- cd8_so[,cd8_so$barcodes %in% sampled_labels$barcodes]

In [ ]:
all_meta <- rbind(all_cd4_meta, all_cd8_meta)

In [ ]:
sampled_so <- merge(cd4_so, cd8_so)

In [ ]:
sampled_so

Split by timepoint

In [ ]:
split_so <- map(
    split_labels,
    function(labels) {
        sampled_so[,labels$barcodes]
    }
)

In [ ]:
split_so <- map(
    split_so,
    function(sampled_so) {
        DefaultAssay(sampled_so) <- "RNA"
        sampled_so <- FindVariableFeatures(sampled_so, nfeatures = 5000)
        sampled_so <- NormalizeData(sampled_so)
        sampled_so <- ScaleData(sampled_so)
        sampled_so <- RunPCA(
            sampled_so, 
            reduction.name = "rnapca",
            reduction.key = "RNAPC_")
        sampled_so
    }
)

### Run Seurat PCA for ADT

In [ ]:
split_so <- map(
    split_so,
    function(sampled_so) {
        DefaultAssay(sampled_so) <- "ADT"
        sampled_so <- NormalizeData(
            sampled_so, 
            normalization.method = "CLR", 
            margin = 2) # normalize across cells
        
        var_feats <- rownames(sampled_so)
        var_veats <- var_feats[!grepl("Control", rownames(sampled_so))]
        VariableFeatures(sampled_so) <- var_feats
        
        sampled_so <- ScaleData(sampled_so)
        sampled_so <- RunPCA(
            sampled_so, 
            npcs = 20, 
            ndims.print = 1,
            reduction.name = "adtpca",
            reduction.key = "ADTPC_")
        
        sampled_so
    }
)

### Add ATAC LSI to Seurat Objects

In [ ]:
split_so <- map2(
    split_so, lsi_mats,
    function(so, lsi_mat) {
        lsi_mat <- lsi_mat[so$barcodes,]
        
        # Spoofing counts using the LSI matrix
        # we don't actually need counts, but we need Seurat to have an ATAC assay.
        
        so[["ATAC"]] <- CreateAssayObject(
            counts = t(lsi_mat)
        )
        
        atac_rd <- CreateDimReducObject(
            embeddings = lsi_mat,
            assay = "ATAC",
            key = "ATACLSI_"
        )
        so@reductions$ataclsi <- atac_rd
        so
    }
)

### Run WNN

In [ ]:
n_lsi <- map(lsi_mats, ncol)

In [ ]:
split_so <- map2(
    split_so, n_lsi,
    function(so, nlsi) {
        FindMultiModalNeighbors(
            so, 
            knn.graph.name = "wknn.tea",
            snn.graph.name = "wsnn.tea",
            weighted.nn.name = "weighted.nn.tea",
            reduction.list = c("rnapca", "adtpca", "ataclsi"),
            dims.list = list(1:30, 1:15, 1:nlsi),
        )
    }
)

### Run UMAP

In [ ]:
split_so <- map(
    split_so,
    function(so) {
        RunUMAP(
            so, 
            nn.name = "weighted.nn.tea", 
            reduction.name = "umap_wnn_tea", 
            reduction.key = "wnnteaUMAP_"
        )
    }
)

### Extract and plot UMAPs

In [ ]:
umap_dfs <- map2(
    split_so, split_labels,
    function(so, labels) {
        umap_df <- as.data.frame(so@reductions$umap_wnn_tea@cell.embeddings)
        names(umap_df) <- c("UMAP_1", "UMAP_2")
        umap_df$barcodes <- rownames(umap_df)
        rownames(umap_df) <- NULL
        
        umap_df <- umap_df %>%
          left_join(labels)
        
        umap_df %>%
          select(barcodes, treatment, timepoint, aifi_cell_type, UMAP_1, UMAP_2)
    }
)

In [ ]:
type_colors <- read.csv("../common/color_sets/aifi_cell_type_colors.csv")
type_colors <- type_colors %>%
  select(aifi_cell_type, cell_type_color)
treat_colors <- read.csv("../common/color_sets/treatment_colors.csv")
treat_colors <- treat_colors %>%
  select(treatment, treatment_color)

In [ ]:
umap_dfs <- map(umap_dfs, left_join, type_colors)
umap_dfs <- map(umap_dfs, left_join, treat_colors)

In [ ]:
walk2(
    umap_dfs, names(umap_dfs),
    function(umap_df, tp) {
        write.csv(
            umap_df,
            file.path(out_dir, paste0("umap_projection_", tp, "_", Sys.Date(), ".csv"))
        )
    }
)

In [ ]:
plot_treat_time <- function(
    plot_df, 
    invx = FALSE, invy = FALSE,
    focus_col = "treatment", focus_grps = NULL,
    drop_bg = FALSE
) {
    if(invx) {
        plot_df$UMAP_1 <- -1 * plot_df$UMAP_1
    }
    if(invy) {
        plot_df$UMAP_2 <- -1 * plot_df$UMAP_2
    }
    
    if(!is.null(focus_grps)) {
        fg_df <- plot_df[plot_df[[focus_col]] %in% focus_grps,]
        bg_df <- plot_df[!plot_df[[focus_col]] %in% focus_grps,]
    } else {
        fg_df <- plot_df
        bg_df <- NULL
    }
    
    p <- ggplot()
    
    if(!is.null(bg_df) & !drop_bg) {
        p <- p + geom_point_rast(
          data = bg_df,
          aes(x = UMAP_1,
              y = UMAP_2,
              color = treatment_color),
          size = 0.1,
          alpha = 0
        )
    }
    
    p <- p +
      geom_point_rast(
          data = fg_df,
          aes(x = UMAP_1,
              y = UMAP_2,
              color = treatment_color),
          size = 0.1
      ) +
      scale_x_continuous("") +
      scale_y_continuous("") +
      scale_color_identity() +
      theme_bw() +
      theme(axis.text = element_blank(),
            axis.ticks = element_blank(),
            panel.grid.minor = element_blank(),
            plot.margin = margin(0,0,0,0))
}
plot_cell_types <- function(
    plot_df, 
    invx = FALSE, invy = FALSE,
    focus_col = "treatment", focus_grps = NULL,
    drop_bg = FALSE
) {
    if(invx) {
        plot_df$UMAP_1 <- -1 * plot_df$UMAP_1
    }
    if(invy) {
        plot_df$UMAP_2 <- -1 * plot_df$UMAP_2
    }
    
    if(!is.null(focus_grps)) {
        fg_df <- plot_df[plot_df[[focus_col]] %in% focus_grps,]
        bg_df <- plot_df[!plot_df[[focus_col]] %in% focus_grps,]
    } else {
        fg_df <- plot_df
        bg_df <- NULL
    }
    
    p <- ggplot()
    
    if(!is.null(bg_df) & !drop_bg) {
        p <- p + geom_point_rast(
          data = bg_df,
          aes(x = UMAP_1,
              y = UMAP_2,
              color = treatment_color),
          size = 0.1,
          alpha = 0
        )
    }
    
    p <- p +
      geom_point_rast(
          data = fg_df,
          aes(x = UMAP_1,
              y = UMAP_2,
              color = cell_type_color),
          size = 0.1
      ) +
      scale_x_continuous("") +
      scale_y_continuous("") +
      scale_color_identity() +
      theme_bw() +
      theme(axis.text = element_blank(),
            axis.ticks = element_blank(),
            panel.grid.minor = element_blank(),
            plot.margin = margin(0,0,0,0))
}

Randomize point order for plots so we don't end up with layering of treatments.

In [ ]:
plot_dfs <- map(
    umap_dfs,
    function(df) {
        df[sample(1:nrow(df), nrow(df)),]
    }
)

In [ ]:
treat_plots <- map(plot_dfs, plot_treat_time)
type_plots <- map(plot_dfs, plot_cell_types)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
all_plots <- plot_grid(
    plotlist = c(treat_plots, type_plots),
    nrow = 2, ncol = 3
)
all_plots

In [ ]:
ggsave(
    file.path(out_dir, paste0("umap_plots_", Sys.Date(), ".pdf")),
    all_plots,
    width = 9, height = 6
)

For other figures, we'll want subsets of these cells displayed. Let's build those for each treatment condition here:

In [ ]:
fg <- c("dmso", "bortezomib")

In [ ]:
treat_plots <- map(plot_dfs, plot_treat_time, focus_grps = fg)
type_plots <- map(plot_dfs, plot_cell_types, focus_grps = fg)

In [ ]:
all_plots <- plot_grid(
    plotlist = c(treat_plots, type_plots),
    nrow = 2, ncol = 3
)
all_plots

In [ ]:
ggsave(
    file.path(out_dir, paste0("umap_plots_bortezomib_", Sys.Date(), ".pdf")),
    all_plots,
    width = 9, height = 6
)

In [ ]:
fg <- c("dmso", "lenalidomide")

In [ ]:
treat_plots <- map(plot_dfs, plot_treat_time, focus_grps = fg, drop_bg = TRUE)
type_plots <- map(plot_dfs, plot_cell_types, focus_grps = fg, drop_bg = TRUE)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
all_plots <- plot_grid(
    plotlist = c(treat_plots, type_plots),
    nrow = 2, ncol = 3
)
all_plots

In [ ]:
ggsave(
    file.path(out_dir, paste0("umap_plots_lenalidomide_", Sys.Date(), ".pdf")),
    all_plots,
    width = 9, height = 6
)

In [ ]:
fg <- c("dmso", "dexamethasone")

In [ ]:
treat_plots <- map(plot_dfs[c("4", "24")], plot_treat_time, focus_grps = fg)
type_plots <- map(plot_dfs[c("4", "24")], plot_cell_types, focus_grps = fg)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 6)
all_plots <- plot_grid(
    plotlist = c(treat_plots, type_plots),
    nrow = 2, ncol = 2
)
all_plots

In [ ]:
ggsave(
    file.path(out_dir, paste0("umap_plots_dexamethasone_", Sys.Date(), ".pdf")),
    all_plots,
    width = 6, height = 6
)

## QC violin Plots

To plot QC values, we'll use all cells per condition so the plots are representative of the full samples.

In [ ]:
tt_pos <- data.frame(
    treat_time = c(
        "dmso_4", "bortezomib_4", "lenalidomide_4", "dexamethasone_4",
        "dmso_24", "bortezomib_24", "lenalidomide_24", "dexamethasone_24",
        "dmso_72", "bortezomib_72", "lenalidomide_72"
    ),
    treat_label = c(
        "DMSO 4 hr", "Bor. 4 hr", "Len. 4 hr", "Dex. 4 hr",
        "DMSO 24 hr", "Bor. 24 hr", "Len. 24 hr", "Dex. 24 hr",
        "DMSO 72 hr", "Bor. 72 hr", "Len. 72 hr"
    ),
    xpos = c(1,2,3,4,
             6,7,8,9,
             11,12,13),
    treatment_color = c(
        "#000000", "#FF9F45", "#59C1BD", "#8D9EFF",
        "#00000C", "#F29344", "#60CCC4", "#889DF2",
        "#000019", "#E58B41", "#54B2AB"
    )
)

In [ ]:
plot_data <- all_meta %>%
  filter(treatment != "untreated") %>%
  mutate(
      treat_time = paste0(treatment, "_", timepoint),
      log_n_genes = log10(n_genes),
      log_n_adt_umis = log10(adt_umis),
      log_n_unique = log10(n_unique)
  ) %>%
  left_join(tt_pos)

In [ ]:
write.csv(
    plot_data, 
    file.path(out_dir, paste0("qc_violin_plots_data_", Sys.Date(), ".csv")),
    row.names = FALSE, quote = FALSE
)

In [ ]:
plot_meds <- plot_data %>%
  group_by(treat_time, xpos) %>%
  dplyr::summarise(
      med_n_genes = median(n_genes),
      med_n_adt_umis = median(adt_umis),
      med_n_unique = median(n_unique),
      med_log_n_genes = median(log_n_genes),
      med_log_n_adt_umis = median(log_n_adt_umis),
      med_log_n_unique = median(log_n_unique),
      .groups = "keep"
  ) %>%
  mutate(color = ifelse(grepl("dmso", treat_time), "white", "black"))

In [ ]:
write.csv(
    plot_meds, 
    file.path(out_dir, paste0("qc_violin_plots_median_values_", Sys.Date(), ".csv")),
    row.names = FALSE, quote = FALSE
)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 3)

genes <- ggplot() +
  geom_violin(
      data = plot_data,
      aes(x = xpos,
          y = log_n_genes,
          fill = treatment_color)
  ) +
  geom_point(
      data = plot_meds,
      aes(x = xpos,
          y = med_log_n_genes,
          color = color)
  ) +
  geom_hline(
      data = plot_meds %>%
        filter(treat_time == "dmso_4"),
      aes(yintercept = med_log_n_genes),
      linetype = "dashed"
  ) +
  geom_text(
      data = plot_meds %>%
        filter(treat_time == "dmso_4"),
      aes(x = 1, y = 3.9, label = med_n_genes)
  ) +
  scale_x_continuous(
      "",
      breaks = tt_pos$xpos,
      labels = tt_pos$treat_label
  ) +
  scale_color_identity() +
  scale_fill_identity() +
  scale_y_continuous("Log10(N Genes)",
                     limits = c(2, 4),
                     breaks = c(2,3,4)) +
  theme_bw() +
  theme(axis.ticks = element_blank(),
        axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.3),
        panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank(),
        panel.grid.minor.y = element_blank())
genes

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 3)
adt <- ggplot() +
  geom_violin(
      data = plot_data,
      aes(x = xpos,
          y = log_n_adt_umis,
          fill = treatment_color)
  ) +
  geom_point(
      data = plot_meds,
      aes(x = xpos,
          y = med_log_n_adt_umis,
          color = color)
  ) +
  geom_text(
      data = plot_meds %>%
        filter(treat_time == "dmso_4"),
      aes(x = 1, y = 3.9, label = med_n_adt_umis)
  ) +
  geom_hline(
      data = plot_meds %>%
        filter(treat_time == "dmso_4"),
      aes(yintercept = med_log_n_adt_umis),
      linetype = "dashed"
  ) +
  scale_x_continuous(
      "",
      breaks = tt_pos$xpos,
      labels = tt_pos$treat_label
  ) +
  scale_color_identity() +
  scale_fill_identity() +
  scale_y_continuous("Log10(ADT UMIs)",
                     limits = c(2.5, 4),
                     breaks = c(2,3,4)) +
  theme_bw() +
  theme(axis.ticks = element_blank(),
        axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.3),
        panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank(),
        panel.grid.minor.y = element_blank())
adt

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 3)
atac <- ggplot() +
  geom_violin(
      data = plot_data,
      aes(x = xpos,
          y = log_n_unique,
          fill = treatment_color)
  ) +
  geom_point(
      data = plot_meds,
      aes(x = xpos,
          y = med_log_n_unique,
          color = color)
  ) +
  geom_text(
      data = plot_meds %>%
        filter(treat_time == "dmso_4"),
      aes(x = 1, y = 4.9, label = med_n_unique)
  ) +
  geom_hline(
      data = plot_meds %>%
        filter(treat_time == "dmso_4"),
      aes(yintercept = med_log_n_unique),
      linetype = "dashed"
  ) +
  scale_x_continuous(
      "",
      breaks = tt_pos$xpos,
      labels = tt_pos$treat_label
  ) +
  scale_color_identity() +
  scale_fill_identity() +
  scale_y_continuous("Log10(Unique Frag.)",
                     limits = c(3, 5),
                     breaks = c(3,4,5)) +
  theme_bw() +
  theme(axis.ticks = element_blank(),
        axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.3),
        panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank(),
        panel.grid.minor.y = element_blank())
atac

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 7)
all_plots <- plot_grid(
    genes, adt, atac,
    ncol = 1
)
all_plots

In [ ]:
ggsave(
    file.path(out_dir, paste0("qc_violin_plots_", Sys.Date(), ".pdf")),
    all_plots,
    width = 6, height = 7
)

## Cell Type Stacked Barplots

In [ ]:
type_colors <- read.csv("../common/color_sets/aifi_cell_type_colors.csv")
type_colors <- type_colors %>%
  select(aifi_cell_type, cell_type_ypos, cell_type_color)

In [ ]:
label_summary <- all_labels %>%
  filter(treatment != "untreated") %>%
  mutate(treat_time = paste0(treatment, "_", timepoint)) %>%
  group_by(treat_time) %>%
  dplyr::mutate(total_n = n()) %>%
  ungroup() %>%
  group_by(treat_time, total_n, aifi_cell_type) %>%
  dplyr::summarise(n_cells = n(),
            frac_cells = n() / total_n[1],
            .groups = "keep")

In [ ]:
plot_data <- label_summary %>%
  left_join(type_colors) %>%
  left_join(tt_pos) %>%
  arrange(cell_type_ypos) %>%
  group_by(treat_time) %>%
  dplyr::mutate(ymax = cumsum(frac_cells)) %>%
  dplyr::mutate(ymin = lag(ymax, default = 0)) %>%
  ungroup()

In [ ]:
write.csv(
    plot_data, 
    file.path(out_dir, paste0("aifi_cell_type_stacked_barplot_data_", Sys.Date(), ".csv")),
    row.names = FALSE, quote = FALSE
)

In [ ]:
options(repr.plot.width = 3, repr.plot.height = 2)

type_barplots <- ggplot() +
  geom_rect(
      data = plot_data,
      aes(xmin = xpos - 0.4,
          xmax = xpos + 0.4,
          ymin = ymin,
          ymax = ymax,
          fill = cell_type_color)
  ) +
  geom_hline(
      data = plot_data %>%
        filter(treat_time == "dmso_4") %>%
        filter(ymax != 1),
      aes(yintercept = ymax),
      linetype = "dashed",
      color = "gray60"
  ) +
  geom_hline(
      data = data.frame(y = c(0,1)),
      aes(yintercept = y)
  ) +
  scale_x_continuous(
      "",
      breaks = tt_pos$xpos,
      labels = tt_pos$treat_label
  ) +
  scale_y_continuous(
      "",
      expand = c(0,0)
  ) +
  scale_fill_identity() +
  theme_bw(base_size = 7) +
  theme(
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.3),
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      panel.grid.minor.y = element_blank(),
      axis.ticks = element_blank(),
      panel.border = element_blank()
  )
type_barplots

In [ ]:
ggsave(
    file.path(out_dir, paste0("aifi_cell_type_stacked_barplot_", Sys.Date(), ".pdf")),
    type_barplots,
    width = 3, height = 2
)

# Store results in HISE

Finally, we store the output file in our Collaboration Space for later retrieval and use. We need to provide the UUID for our Collaboration Space (aka `studySpaceId`), as well as a title for this step in our analysis process.

The hise function `uploadFiles()` also requires the FileIDs from the original fileset for reference.

In [ ]:
study_space_uuid <- "40df6403-29f0-4b45-ab7d-f46d420c422e"
title <- paste("VRd TEA-seq Figure 1", Sys.Date())

In [ ]:
out_files <- list.files(
    "output/figure_1",
    full.names = TRUE
)
out_list <- as.list(out_files)

In [ ]:
out_list

In [ ]:
in_list <- c(
    so_uuids, label_uuids, arrow_uuids
)

In [ ]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    store = "project",
    doPrompt = FALSE
)

In [ ]:
sessionInfo()